# 🪐 Astrophage — Two-Stage Random Forest for Exoplanet Classification

> **Celesta — India High School Exoplanet Data Challenge 2026**
>
> This notebook runs the Astrophage Rust pipeline in Google Colab. No local installation needed!

---

## What is Astrophage?

Astrophage is a custom Two-Stage Random Forest classifier built in **Rust** using **Polars** that classifies NASA Kepler Objects of Interest (KOIs) into three categories:

- ✅ **CONFIRMED** — Validated exoplanets
- 🔍 **CANDIDATE** — Promising signals awaiting confirmation
- ❌ **FALSE POSITIVE** — Non-planetary signals (stellar binaries, noise, etc.)

### Architecture
```
Stage 1: CONFIRMED vs NOT CONFIRMED  (easy separation)
         ↓
Stage 2: CANDIDATE vs FALSE POSITIVE  (hard separation)
```

This mirrors NASA's actual vetting workflow and achieves **94.81% accuracy**.

---

## 🚀 Quick Start (Run All Cells)

### Step 1: Install Rust

Google Colab doesn't come with Rust pre-installed. We'll use `rustup` to install it.

> ⏱️ **Takes ~2-3 minutes**

In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential curl
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!rustc --version
!cargo --version

### Step 2: Clone the Astrophage Repository

In [ ]:
!git clone https://github.com/harihar-nautiyal/astrophage.git
%cd astrophage
!ls -la

### Step 3: Build the Project

> ⏱️ **Takes ~3-5 minutes** (Rust compiles everything from scratch)

The project uses:
- `polars` — Fast DataFrame operations
- `ndarray` — N-dimensional arrays for ML math
- `tokio` — Async runtime
- `serde` — JSON serialization
- Custom decision tree & random forest implementations

In [ ]:
!cargo build --release 2>&1 | tail -20

### Step 4: Run the Full Pipeline

This executes the complete two-stage classification:
1. Load KOI dataset
2. Engineer 36 features (28 base + 8 derived)
3. Stratified 80/20 train/test split
4. Train Stage 1 (CONFIRMED vs NOT)
5. Train Stage 2 (CANDIDATE vs FALSE_POSITIVE)
6. Evaluate and generate report

In [ ]:
!./target/release/astrophage

---

## 📊 Results & Visualization

Let's parse the generated `report.json` and visualize the results.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the report
with open('output/report.json', 'r') as f:
    report = json.load(f)

print("🪐 ASTROPHAGE REPORT v{}".format(report['version']))
print("=" * 50)
print(f"Total Samples: {report['summary']['total_samples']}")
print(f"Classes: {report['summary']['n_classes']}")
print(f"Model Type: {report['summary']['model_type']}")
print()

### Overall Metrics

In [ ]:
metrics = report['metrics']
print(f"Accuracy:  {metrics['accuracy']:.4f}  ({metrics['accuracy']*100:.2f}%)")
print(f"Macro F1:  {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")

### Per-Class Performance

In [ ]:
per_class = metrics['per_class']
df = pd.DataFrame(per_class).T
df.columns = ['Precision', 'Recall', 'F1-Score']
df = df.round(4)
display(df)

# Plot
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
classes = list(per_class.keys())
x = np.arange(len(classes))

for i, metric in enumerate(['precision', 'recall', 'f1_score']):
    values = [per_class[c][metric] for c in classes]
    ax[i].bar(classes, values, color=['#2ecc71', '#e74c3c', '#3498db'])
    ax[i].set_ylim(0, 1.05)
    ax[i].set_title(metric.replace('_', ' ').title())
    ax[i].set_ylabel('Score')
    for j, v in enumerate(values):
        ax[i].text(j, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Feature Importance (Top 15)

In [ ]:
features = report['feature_importance'][:15]
names = [f['feature_name'] for f in features]
scores = [f['importance_score'] for f in features]
meanings = [f['astrophysical_meaning'] for f in features]

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(names))
bars = ax.barh(y_pos, scores, color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Feature Importances — Astrophage Two-Stage RF', fontsize=14, fontweight='bold')

# Add score labels
for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{score:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print meanings
print("\n📖 Astrophysical Meanings:")
print("=" * 60)
for f in features[:10]:
    print(f"\n{f['rank']:2d}. {f['feature_name']}")
    print(f"    {f['astrophysical_meaning']}")

### Astrophysical Insights

In [ ]:
insights = report['astrophysical_insights']

for i, ins in enumerate(insights, 1):
    confidence = ins['confidence']
    emoji = '🔴' if 'Very High' in confidence else '🟡' if 'High' in confidence else '🟢'
    print(f"{emoji} Insight {i} [{confidence}]")
    print(f"   {ins['insight']}")
    print(f"   Supporting features: {', '.join(ins['supporting_features'])}")
    print()

### Recommendations

In [ ]:
print("💡 Recommendations from the Model:")
print("=" * 50)
for rec in report['recommendations']:
    print(f"  • {rec}")

---

## 🧪 Experiment: Replicate in Python (sklearn)

For comparison, here is how the same two-stage logic would look in Python using scikit-learn. This is useful for understanding the architecture without diving into Rust.

In [ ]:
# This is a conceptual Python equivalent for educational purposes
# The actual Rust implementation is fully custom (no sklearn dependency!)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

print('Two-Stage Random Forest (Python Conceptual Equivalent)')
print('=' * 60)
print()
print('# STAGE 1: CONFIRMED vs NOT CONFIRMED')
print('stage1 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)')
print('stage1.fit(X_train, y_train_binary)  # y: CONFIRMED=1, NOT=0')
print()
print('# STAGE 2: CANDIDATE vs FALSE_POSITIVE (only on NOT CONFIRMED)')
print('stage2 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)')
print('stage2.fit(X_train_stage2, y_train_stage2)  # y: CANDIDATE=1, FALSE_POSITIVE=0')
print()
print('# Inference:')
print('# 1. Run Stage 1 -> if CONFIRMED, output CONFIRMED')
print('# 2. If NOT CONFIRMED, run Stage 2 -> output CANDIDATE or FALSE_POSITIVE')
print()
print('The Rust implementation achieves 94.81% accuracy with this architecture!')

---

## 📁 Download Results

You can download the generated `report.json` to your local machine:

In [ ]:
from google.colab import files
files.download('output/report.json')

---

## 🔗 Links

- **GitHub Repository:** [harihar-nautiyal/astrophage](https://github.com/harihar-nautiyal/astrophage)
- **Hackathon:** [Celesta — India High School Exoplanet Data Challenge 2026](https://celesta-exoplanet-challenge.devpost.com/)
- **Author:** [@harihar-nautiyal](https://github.com/harihar-nautiyal)

---

<p align="center">
  <i>"Somewhere, something incredible is waiting to be known."</i><br>
  — Carl Sagan
</p>